> ### Note on Labs and Assignments:
>
> 🔧 Look for the **wrench emoji** 🔧 — it highlights where you're expected to take action!
>
> These sections are graded and are not optional.
>

# IS 4487 Lab 10: Classification Trees with SF Rent Data

## Outline

- Define a binary classification target based on rental price  
- Split the dataset into training and testing sets  
- Fit a Decision Tree Classifier  
- Visualize the decision tree structure  
- Evaluate model accuracy and feature importance  
- Fit and evaluate a Random Forest Classifier
- Fit and evaluate a Gradient Boosting Classifier
- Compare all three classification models

In this lab, we’ll build a classification model to predict whether a rental listing in San Francisco is considered **"affordable"** based on its features.

We are continuing with the **SF Rent dataset** used in **Lab 9**, and will explore how classification trees can uncover patterns in rental prices.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Labs/lab_10_classification.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>



## Dataset Overview

**Dataset:** `rent.csv`  
Source: [TidyTuesday-2022-7-05](https://github.com/rfordatascience/tidytuesday/blob/main/data/2022/2022-07-05/rent.csv)

| Variable       | Type       | Description |
|----------------|------------|-------------|
| `post_id`      | Categorical| Unique listing ID |
| `date`         | Numeric    | Listing date (numeric format) |
| `year`         | Integer    | Year of listing |
| `nhood`        | Categorical| Neighborhood |
| `city`         | Categorical| City |
| `county`       | Categorical| County |
| `price`        | Numeric    | Listing price (USD) |
| `beds`         | Numeric    | Number of bedrooms |
| `baths`        | Numeric    | Number of bathrooms |
| `sqft`         | Numeric    | Square footage |
| `room_in_apt`  | Binary     | 1 = room in apartment |
| `address`      | Categorical| Street address |
| `lat`          | Numeric    | Latitude |
| `lon`          | Numeric    | Longitude |
| `title`        | Text       | Listing title |
| `descr`        | Text       | Listing description |
| `details`      | Text       | Additional details |

## Part 1: Clean and Prepare the Data

We'll repeat the cleaning process from Lab 9, including:
- Removing duplicates
- Dropping missing values in key columns
- Filtering out extreme outliers
- Converting data types

### Why This Matters:
Good models depend on clean, reliable inputs.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
# Load dataset
url = 'https://raw.githubusercontent.com/Stan-Pugsley/is_4487_base/refs/heads/main/DataSets/rent.csv'
df = pd.read_csv(url)

# Initial check
df.info()

In [ ]:
# Drop duplicate listings using post_id
df = df.drop_duplicates(subset='post_id')

# Drop rows missing essential features
required_cols = ['price', 'beds', 'baths', 'sqft', 'lat', 'lon']
df = df.dropna(subset=required_cols)

# Remove outliers
df = df[df['price'].between(500, 20000)]
df = df[df['beds'].between(0, 10)]
df = df[df['baths'].between(0.5, 10)]
df = df[df['sqft'].between(100, 5000)]

# Convert columns to appropriate data types
df['beds'] = df['beds'].astype(int)
df['baths'] = df['baths'].astype(float)
df['sqft'] = df['sqft'].astype(int)
df['price'] = df['price'].astype(int)

# Reset index
df = df.reset_index(drop=True)

# Quick check
df.info()

In [ ]:
# Preview data
df.head()

## Part 2: Create a Binary Target Variable

We'll classify whether a listing is **Affordable** (price < 3,000).

- 1 = Affordable
- 0 = Not Affordable

### Why This Matters:
To use classification trees, we need a target variable with a limited number of classes.


In [ ]:
# Create binary target variable
df['affordable'] = (df['price'] < 3000).astype(int)

# Check target class distribution
df['affordable'].value_counts(normalize=True)


### 🔧 Do the following – Part 2

1. Adjust the affordable threshold to $5,000 and check the class balance again  
2. Calculate what percentage of listings are labeled as affordable



In [ ]:
# Add code here

Reset the threshold to $3000

In [ ]:
# Reset binary target back to 3000
df['affordable'] = (df['price'] < 3000).astype(int)

## Part 3: Select Features and Split Data

We'll use property characteristics as predictors:
- `beds`, `baths`, `sqft`

Then split the data into 80% training and 20% testing.

### Why This Matters:
Train/test splitting ensures our model is evaluated on unseen data.

*Note*: Stratify tells the split to preserve the same class balance in both the training and testing datasets. This is especially important in classification problems when one class is much more common than the other. Without stratification, the test set could accidentally contain too few examples of one class, making the model evaluation less reliable.

In [ ]:
# Select features and target
X = df[['beds', 'baths', 'sqft']]
y = df['affordable']

# Create train/test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Check dimensions
X_train.shape, X_test.shape


### 🔧 Reflection - Part 3: 
1. What would be the impact if we added `price` as an independent variable?

### ✍️ Your Response: 🔧
1. 

## Part 4: Build and Fit the Decision Tree Classifier

We'll train a `DecisionTreeClassifier` on our data.

### Why This Matters:
Decision trees make predictions by splitting data into branches using rules that reduce impurity.


In [ ]:
# Initialize classifier
clf = DecisionTreeClassifier(random_state=42)

# Train model on training data
clf.fit(X_train, y_train)

### Visualize the tree
Pay close attention to how wide and deep the branches reach

In [ ]:
# Plot decision tree
plt.figure(figsize=(14, 8))
plot_tree(clf, feature_names=X.columns, class_names=['Not Affordable', 'Affordable'], filled=True)
plt.title("Decision Tree for Affordable Listings")
plt.show()


### 🔧 Do the following – Part 4

1. Create a second model called `clf_limited` with `max_depth=3`  
2. Fit it on the same training data 
3. Visualize this new tree  
> You’ll compare the accuracy of this smaller tree in the next section


In [ ]:
# 🔧 Add code here

## 5. Evaluate the Model

We’ll check how well the tree performed on the test data using:
- Accuracy score
- Confusion matrix

### Why This Matters:
We want to know how often the model is right and what kind of mistakes it makes.


In [ ]:
# Predict on test set
y_pred = clf.predict(X_test)

# Accuracy
acc = accuracy_score(y_test, y_pred)
print("Test Accuracy:", acc)

# Confusion matrix
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


### 🔧 Do the following – Part 5

1. Use the `clf_limited` model you created in Part 4 to predict on the test set  
2. Calculate and print the **test accuracy** for `clf_limited`

In [ ]:
# 🔧 Add code here


### Reflection: 
1. Compare the test accuracy of `clf_limited` to the original `clf`. Which performs better?

### ✍️ Your Response: 🔧
1. 

## Part 6: Evaluate the Tree Feature Importance

We’ll rank the most important predictors.

### Why This Matters:
Tree plots explain the model's logic. Feature importance shows what drives predictions.


In [ ]:
# Plot feature importance
feat_imp = pd.Series(clf.feature_importances_, index=X.columns)
sns.barplot(x=feat_imp.values, y=feat_imp.index)
plt.title("Feature Importance Scores")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()


### 🔧 Do the following – Part 6

1. Evaluate the importance of features in the smaller tree, `clf_limited`   

In [ ]:
# 🔧 Add code here


### Reflection: 
1. Which feature does the limitted tree split on first?  
2. Is the structure simpler or more complex than the original `clf` tree?

### ✍️ Your Response: 🔧
1. 
2. 

## Part 7: Random Forest Classification
A random forest combines predictions from many decision trees. Each tree is trained on a slightly different sample of the data and considers a random subset of features at each split.
### Why This Matters:
A single decision tree can overfit the training data. Random forests often provide more stable and accurate predictions by combining many trees.

In [ ]:
# Create a random forest model
rf_clf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

# Train the model
rf_clf.fit(X_train, y_train)

# Predict on the test data
rf_pred = rf_clf.predict(X_test)

# Evaluate accuracy
rf_acc = accuracy_score(y_test, rf_pred)
print("Random Forest Test Accuracy:", rf_acc)

# Display confusion matrix
print("Confusion Matrix:\n", confusion_matrix(y_test, rf_pred))

### Feature Importance in the Random Forest

In [ ]:
# Create and plot feature importance scores
rf_feat_imp = pd.Series(rf_clf.feature_importances_, index=X.columns)

sns.barplot(x=rf_feat_imp.values, y=rf_feat_imp.index)
plt.title("Random Forest Feature Importance Scores")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

### 🔧 Do the following – Part 7
1. Create a second random forest model called rf_limited.
2. Set n_estimators=50 and max_depth=3.
3. Fit the model and calculate its test accuracy.
4. Compare its accuracy to the original random forest.

In [ ]:
# 🔧 Add code here

### Reflection:
1. Why might a random forest perform better than one decision tree?
2. Does limiting the depth of each tree appear to improve or reduce test accuracy?
### ✍️ Your Response: 🔧
1.
2.

## Part 8: Gradient Boosting Classification
Gradient boosting is another ensemble method. Unlike a random forest, which builds many independent trees, gradient boosting builds trees one at a time. Each new tree focuses on improving errors made by the earlier trees.
### Why This Matters:
Gradient boosting can create a strong predictive model from many small, simple trees. However, it may require careful tuning to avoid overfitting.

In [ ]:
# Create a gradient boosting model
gb_clf = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

# Train the model
gb_clf.fit(X_train, y_train)

# Predict on the test data
gb_pred = gb_clf.predict(X_test)

# Evaluate accuracy
gb_acc = accuracy_score(y_test, gb_pred)
print("Gradient Boosting Test Accuracy:", gb_acc)

# Display confusion matrix
print("Confusion Matrix:\n", confusion_matrix(y_test, gb_pred))

### Feature Importance in Gradient Boosting

In [ ]:
# Create and plot feature importance scores
gb_feat_imp = pd.Series(gb_clf.feature_importances_, index=X.columns)

sns.barplot(x=gb_feat_imp.values, y=gb_feat_imp.index)
plt.title("Gradient Boosting Feature Importance Scores")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

### 🔧 Do the following – Part 8
1. Create a second gradient boosting model called gb_slow.
2. Set learning_rate=0.05 and n_estimators=200.
3. Fit the model and calculate its test accuracy.
4. Compare the result to the original gradient boosting model.

In [ ]:
# 🔧 Add code here

### Reflection:
1. What is the tradeoff between using a smaller learning rate and more trees?
2. Which feature is most important in the gradient boosting model?
3. Is the most important feature consistent across the decision tree, random forest, and gradient boosting models?
### ✍️ Your Response: 🔧
1.
2.
3.

## Part 9: Compare Model Performance
Now, compare the accuracy of the four main models:
- Full decision tree
- Limited-depth decision tree
- Random forest
- Gradient boosting

In [ ]:
# Create a comparison table
model_results = pd.DataFrame({
    'Model': [
        'Decision Tree',
        'Limited Decision Tree',
        'Random Forest',
        'Gradient Boosting'
    ],
    'Test Accuracy': [
        acc,
        accuracy_score(y_test, clf_limited.predict(X_test)),
        rf_acc,
        gb_acc
    ]
})

model_results.sort_values(by='Test Accuracy', ascending=False)

### 🔧 Do the following – Part 9
1. Add your rf_limited and gb_slow models to the comparison table.
2. Identify the model with the highest test accuracy.
3. Create a bar chart showing the test accuracy of each model.

In [ ]:
# 🔧 Add code here

## 🔧 Part 10: Final Reflection (100 words or less per question)
1. Which model had the best test accuracy? Does that automatically make it the best model for every business situation?
2. Compare a decision tree, random forest, and gradient boosting model in terms of interpretability. Which would be easiest to explain to a landlord or property manager?
3. Would you trust one of these models to recommend rental prices to landlords? What additional variables or safeguards would you want before using it in practice?



### ✍️ Your Response: 🔧
1. 

## Export Your Notebook to Submit in Canvas
- Use the instructions from Lab 1

In [ ]:
!jupyter nbconvert --to html "lab_10_classification.ipynb"